In [ ]:
!pip install torch tqdm datasets transformers peft trl accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model
from accelerate import Accelerator
from torch.utils.data import DataLoader

accelerator = Accelerator()
device = accelerator.device

MODEL = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16
)

# LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# DATA
dataset = load_dataset("gsm8k", "main", split="train").select(range(2000))

def format_example(x):
    q = x["question"]
    full = x["answer"]
    final = full.split("####")[-1].strip()

    return {
        "text": f"""Solve the problem.

Question: {q}

<think>
{full}
</think>
<answer>{final}</answer>
"""
    }

dataset = dataset.map(format_example)

def tokenize(x):
    return tokenizer(x["text"], truncation=True, padding="max_length", max_length=512)

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

loader = DataLoader(dataset, batch_size=2, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

model, optimizer, loader = accelerator.prepare(model, optimizer, loader)

model.train()

for epoch in range(1):
    for step, batch in enumerate(loader):
        outputs = model(**batch, labels=batch["input_ids"])
        loss = outputs.loss

        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()

        if step % 50 == 0 and accelerator.is_main_process:
            print("loss:", loss.item())

if accelerator.is_main_process:
    model.save_pretrained("phi3_sft_fixed")
    tokenizer.save_pretrained("phi3_sft_fixed")

In [ ]:
import shutil

# Zip phi3_sft_lora
shutil.make_archive(
    '/kaggle/working/phi3_sft_fixed', 
    'zip', 
    '/kaggle/working/phi3_sft_fixed'
)


print("✅ Zipping done!")

In [ ]:
import torch
import re
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =========================
# CONFIG
# =========================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
LORA_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr"   # <-- change if needed

MAX_NEW_TOKENS = 120
NUM_SAMPLES = 50

# =========================
# LOAD MODEL
# =========================

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

# =========================
# PROMPT (FIXED VERSION)
# =========================

def build_prompt(question, task="math"):
    if task == "math":
        return f"""Solve the problem.

Question: {question}

IMPORTANT:
- Put reasoning in <think>
- Put final answer in <answer>
"""
    elif task == "qa":
        return f"""Answer yes or no.

Question: {question}

Use:
<think>...</think>
<answer>yes/no</answer>
"""
    elif task == "mcq":
        return f"""Choose correct option.

{question}

Answer format:
<answer>A/B/C/D</answer>
"""

# =========================
# GENERATE
# =========================

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# =========================
# EXTRACTION (ROBUST)
# =========================

def extract_number(text):
    # try structured
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    if match:
        nums = re.findall(r"[-+]?\d*\.\d+|\d+", match.group(1))
        if nums:
            return nums[-1]

    # fallback
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", text)
    return nums[-1] if nums else None


def extract_yes_no(text):
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    if match:
        ans = match.group(1).lower()
        if "yes" in ans:
            return "yes"
        if "no" in ans:
            return "no"

    # fallback
    text = text.lower()
    if "yes" in text:
        return "yes"
    if "no" in text:
        return "no"

    return None


def extract_mcq(text):
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    if match:
        ans = match.group(1).upper()
        for c in ["A", "B", "C", "D"]:
            if c in ans:
                return c

    for c in ["A", "B", "C", "D"]:
        if c in text:
            return c

    return None

# =========================
# GSM8K
# =========================

def run_gsm8k(dataset):
    rows = []

    for i, sample in enumerate(tqdm(dataset)):
        q = sample["question"]
        gt = sample["answer"].split("####")[-1].strip()

        out = generate(build_prompt(q, "math"))
        pred = extract_number(out)

        correct = False
        try:
            correct = float(pred) == float(gt)
        except:
            pass

        if i < 3:
            print("\n--- SAMPLE ---")
            print("Q:", q)
            print("OUT:", out)
            print("PRED:", pred, "GT:", gt)

        rows.append({
            "question": q,
            "ground_truth": gt,
            "prediction": pred,
            "correct": correct
        })

    df = pd.DataFrame(rows)
    acc = df["correct"].mean()

    df.to_csv("gsm8k_sft.csv", index=False)
    print("\nGSM8K Accuracy:", acc)

    return acc

# =========================
# STRATEGYQA
# =========================

def run_strategyqa(dataset):
    rows = []

    for sample in tqdm(dataset):
        q = sample["question"]
        gt = "yes" if sample["answer"] else "no"

        out = generate(build_prompt(q, "qa"))
        pred = extract_yes_no(out)

        rows.append({
            "question": q,
            "ground_truth": gt,
            "prediction": pred,
            "correct": pred == gt
        })

    df = pd.DataFrame(rows)
    acc = df["correct"].mean()

    df.to_csv("strategyqa_sft.csv", index=False)
    print("\nStrategyQA Accuracy:", acc)

    return acc

# =========================
# MMLU
# =========================

def run_mmlu(dataset):
    rows = []

    for sample in tqdm(dataset):
        q = sample["question"]
        choices = sample["choices"]
        gt = chr(65 + sample["answer"])

        formatted_q = q + "\n"
        for i, c in enumerate(choices):
            formatted_q += f"{chr(65+i)}. {c}\n"

        out = generate(build_prompt(formatted_q, "mcq"))
        pred = extract_mcq(out)

        rows.append({
            "question": q,
            "ground_truth": gt,
            "prediction": pred,
            "correct": pred == gt
        })

    df = pd.DataFrame(rows)
    acc = df["correct"].mean()

    df.to_csv("mmlu_sft.csv", index=False)
    print("\nMMLU Accuracy:", acc)

    return acc

# =========================
# LOAD DATA
# =========================

gsm8k = load_dataset("gsm8k", "main", split="test").select(range(NUM_SAMPLES))
strategyqa = load_dataset("ChilleD/StrategyQA", split="test").select(range(NUM_SAMPLES))
mmlu = load_dataset("cais/mmlu", "abstract_algebra", split="test").select(range(NUM_SAMPLES))

# =========================
# RUN
# =========================

gsm_acc = run_gsm8k(gsm8k)
strat_acc = run_strategyqa(strategyqa)
mmlu_acc = run_mmlu(mmlu)

print("\nFINAL RESULTS")
print("GSM8K:", gsm_acc)
print("StrategyQA:", strat_acc)
print("MMLU:", mmlu_acc)